# Encapsulation in Python

Goal: understand why encapsulation exists, public/protected/private naming conventions, name mangling, and controlling access with `@property`.

## 1. What is Encapsulation?

**Why we need it:** if every attribute on an object can be read and overwritten from anywhere, nothing stops other code from setting a `BankAccount`'s `balance` to `-500` directly, or a `Temperature`'s `celsius` to `"hot"`. The object itself has no way to protect its own data — any invariant you cared about ("balance can't go negative", "celsius is always a number") can be silently broken from outside.

**Use case:** anywhere an object's internal data needs to stay valid/consistent, or where you want the freedom to change *how* something is stored internally later without breaking every piece of code that uses the class — a `BankAccount` guarding its `balance`, a `Temperature` class that stores Celsius internally but exposes Fahrenheit too, a config object that validates values before accepting them.

**How to implement:** bundle data and the methods that operate on it together inside a class (you've already been doing this since the first notebook), and control access to the sensitive parts — by convention marking what's "internal" with a naming convention, and by using `@property` to run code whenever a value is read or written, instead of allowing direct, unchecked access.

## 2. Public, Protected, and Private — by Convention

**Why we need it:** Python has no hard "keep out" wall around attributes the way some other languages do — you can technically reach into any object and read or set anything. What Python has instead is a naming convention: a way of signaling *intent* to other developers (and to yourself, six months later) about which attributes are meant to be used from outside the class and which aren't.

**Use case:** every class has some attributes meant to be part of its public interface (safe to read/set from outside) and some that are implementation detail (subject to change, not meant to be touched directly) — this convention documents which is which, right in the attribute's name.

**How to implement:**
- `self.name` — **public**. No leading underscore. Freely used from outside the class.
- `self._name` — **protected**, by convention. One leading underscore signals "internal — please don't touch this from outside the class (or outside a subclass), even though Python won't stop you."
- `self.__name` — **private**, by convention, and additionally *name-mangled* by Python (covered in the next section) to make accidental access from outside noticeably harder.

Nothing here is enforced by the language — a single leading underscore is purely a social contract. It works because Python culture takes it seriously.

In [6]:
class Demo:
    def __init__(self):
        self.public = "anyone can touch this"
        self._protected = "please don't touch this from outside"
        self.__private = "really shouldn't touch this from outside"

    def show_private(self):
        return self.__private   # fine: accessed from inside the class


d = Demo()
print(d.public)                 # works, as expected
print(d._protected)             # ALSO works -- Python doesn't stop you, it's just a convention
print(d.show_private())         # accessing __private from inside the class works fine

try:
    print(d.__private)          # this is where it gets interesting
except AttributeError as e:
    print(f"AttributeError: {e}")

anyone can touch this
please don't touch this from outside
really shouldn't touch this from outside
AttributeError: 'Demo' object has no attribute '__private'


## 3. Name Mangling

**Why we need it:** the `AttributeError` above needs an explanation — `d.__private` failed even though `self.__private` was set inside `__init__`. That's not a coincidence and not a typo; it's Python actively rewriting the attribute's name behind the scenes specifically to make outside access clumsy on purpose.

**Use case:** mainly this matters when you're debugging exactly the situation above — "I set this attribute, why can't I read it back the same way?" — and, more usefully, it avoids a subclass accidentally overwriting a double-underscore attribute a parent class relies on internally, since the mangled name is different per class.

**How to implement (or rather, how it already works):** any attribute name inside a class body that starts with two underscores and doesn't end with two underscores (so `__private`, but not `__init__`) gets automatically rewritten by Python to `_ClassName__private`. `self.__private = ...` inside `Demo` actually creates `self._Demo__private`. That's why `d.__private` fails from outside — that exact name was never created — while `d._Demo__private` still works, because that's the real, mangled name.

In [3]:
print(vars(d))                    # shows the REAL attribute names Python stored
print(d._Demo__private)           # still reachable -- "private" is a deterrent, not a lock

# name mangling is per-class, so a subclass's own __hidden doesn't collide with the parent's
class Sub(Demo):
    def __init__(self):
        super().__init__()
        self.__hidden = "Sub's own hidden"   # becomes self._Sub__hidden, NOT self._Demo__hidden


s = Sub()
print(vars(s))

{'public': 'anyone can touch this', '_protected': "please don't touch this from outside", '_Demo__private': "really shouldn't touch this from outside"}
really shouldn't touch this from outside
{'public': 'anyone can touch this', '_protected': "please don't touch this from outside", '_Demo__private': "really shouldn't touch this from outside", '_Sub__hidden': "Sub's own hidden"}


## 4. Controlling Access with `@property`

**Why we need it:** naming conventions only ever *ask* people not to touch an attribute — they don't stop `account.balance = -500` from happening. If an attribute genuinely needs to be validated on every write, or computed on every read, a plain attribute can't do that; you need code to run in between.

**Use case:** rejecting invalid values (a negative balance, an empty name), computing a derived value on the fly instead of storing it redundantly (`fahrenheit` computed from a stored `celsius`), or making an attribute read-only from outside.

**How to implement:** store the real value in a conventionally private attribute (`self._balance`), then expose a **property** with the public name using `@property` on a method that returns it — now `account.balance` looks and reads exactly like a plain attribute, but it's actually running a method. Pair it with `@balance.setter` on a method of the same name to control what happens on `account.balance = ...` — that's where you put validation. If you only write the `@property` getter and skip the setter, the attribute becomes effectively read-only from outside: attempting to assign to it raises an `AttributeError`.

In [4]:
class Temperature:
    def __init__(self, celsius):
        self._celsius = celsius     # real storage lives in a protected attribute

    @property
    def celsius(self):              # getter: account.celsius reads like a plain attribute
        return self._celsius

    @celsius.setter
    def celsius(self, value):       # setter: account.celsius = ... runs this, with validation
        if not isinstance(value, (int, float)):
            raise TypeError("celsius must be a number")
        self._celsius = value

    @property
    def fahrenheit(self):           # read-only: computed on the fly, no setter defined
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
print(t.celsius, t.fahrenheit)      # both read like plain attributes

t.celsius = 30                      # goes through the setter -> validated
print(t.celsius, t.fahrenheit)      # fahrenheit is automatically up to date

try:
    t.celsius = "hot"               # setter rejects this
except TypeError as e:
    print(f"TypeError: {e}")

try:
    t.fahrenheit = 100               # no setter defined -> read-only
except AttributeError as e:
    print(f"AttributeError: {e}")

25 77.0
30 86.0
TypeError: celsius must be a number
AttributeError: property 'fahrenheit' of 'Temperature' object has no setter


## 5. Putting It Together: a guarded `BankAccount`

**Why we need it:** the opening example of this notebook was "nothing stops `account.balance = -500`" — here's the actual fix, combining a protected attribute with a validating property, plus methods that are the *only* sanctioned way to change the balance.

**Use case:** a `BankAccount` where the balance must never go negative and must always be a number, no matter which piece of code is interacting with it.

**How it's implemented here:** the real value lives in `self._balance`. The public `balance` property exposes it for reading; its setter rejects invalid or negative values, so even a direct `account.balance = ...` assignment can't break the invariant. `deposit`/`withdraw` go through `self.balance = ...` too, so they automatically get the same validation instead of a second, separate copy of it.

In [5]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance      # goes through the setter below, even here in __init__

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError("balance must be a number")
        if value < 0:
            raise ValueError("balance cannot be negative")
        self._balance = value

    def deposit(self, amount):
        self.balance = self.balance + amount    # reuses the same validated setter

    def withdraw(self, amount):
        self.balance = self.balance - amount    # negative result is rejected automatically


acc = BankAccount("Alice", 100)
acc.deposit(50)
print(acc.balance)

try:
    acc.withdraw(1000)          # would make balance negative -> setter rejects it
except ValueError as e:
    print(f"ValueError: {e}")

try:
    acc.balance = -500          # direct assignment is guarded too, not just deposit/withdraw
except ValueError as e:
    print(f"ValueError: {e}")

print(acc.balance)               # unchanged -- still 150

150
ValueError: balance cannot be negative
ValueError: balance cannot be negative
150
